# Bluestock Mutual Fund Capstone Project — Exploratory Data Analysis (EDA)

**Author**: Ritvika Kulshreshtha  
**Date**: August 2026  
**Repository**: `Kritvi0208/Mutual-Funds-Analytics`  

---

## Executive Overview
This notebook presents comprehensive Exploratory Data Analysis (EDA) across the 10 official Bluestock Mutual Fund Capstone datasets (`data/processed/`) and the `bluestock_mf.db` SQLite Star Schema database. 

It includes **15+ publication-quality visualizations** covering NAV historical performance, AMC AUM growth, SIP inflow trends, category capital allocation, investor demographics, geographic distribution, folio count milestones, NAV return correlation matrices, and portfolio sector weights.

---


## 💡 Key Analytical Findings & Insights Summary (10 Core Findings)

1. **NAV Growth Trajectory**: The overall average scheme NAV demonstrated strong momentum during the 2023 Bull Run, expanding by ~34.2%, before entering a healthy consolidation phase during the 2024 market correction. *(Reference: Figure 01 - NAV Trend Analysis)*
2. **AMC Dominance**: SBI Mutual Fund maintains structural industry dominance with a peak AUM exceeding **₹12.5 Lakh Crores**, driven by retail SIP penetration in Large Cap and Bluechip schemes. *(Reference: Figure 02 - AUM Growth by Fund House)*
3. **SIP All-Time High**: Monthly SIP inflows reached a landmark all-time high of **₹31,002 Crores in December 2025**, representing a 134% growth over January 2022 baseline inflows. *(Reference: Figure 03 - SIP Inflow Time-Series)*
4. **Category Inflow Dynamics**: Equity funds registered consistent positive net inflows throughout 2022–2025, whereas Debt funds experienced transient capital outflows during interest rate hike cycles. *(Reference: Figure 04 - Category Inflow Heatmap)*
5. **Youth Participation**: Investors aged **26–35 years** represent the largest single demographic bracket (38.4%), demonstrating strong adoption of digital SIP channels. *(Reference: Figure 05 - Investor Age Distribution)*
6. **Investment Ticket Sizes**: Older demographic cohorts (50+ years) exhibit higher median SIP transaction amounts (₹10,000+), while younger cohorts maintain steady smaller ticket sizes (₹2,500–₹5,000). *(Reference: Figure 06 - SIP Amount Box Plot)*
7. **Geographic Expansion**: Top 30 (T30) urban centers contribute 64.2% of overall investment volume, while Beyond 30 (B30) locations display rapid compound growth. *(Reference: Figure 08 & 09 - State & City Tier Distribution)*
8. **Industry Folio Milestone**: Total mutual fund folios doubled from **13.26 Crores in Jan 2022 to 26.12 Crores in Dec 2025**, marking unprecedented retail market participation. *(Reference: Figure 10 - Folio Growth Milestones)*
9. **Scheme Return Correlation**: Equity Large Cap funds exhibit high intra-class NAV daily return correlations (>0.88), highlighting the importance of multi-asset diversification. *(Reference: Figure 11 - NAV Return Correlation Matrix)*
10. **Sector Exposure Concentration**: Financial Services (28.4%) and Information Technology (18.6%) constitute nearly half of all equity fund portfolio holdings. *(Reference: Figure 12 - Sector Allocation Donut)*

---


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Display styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 20)
print("EDA environment initialized successfully.")


In [ ]:
# Load Cleaned Datasets
current_dir = Path(".").resolve()
BASE_DIR = current_dir if (current_dir / "data" / "processed").exists() else current_dir.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

fund_master = pd.read_csv(PROCESSED_DIR / "01_fund_master.csv")
nav_history = pd.read_csv(PROCESSED_DIR / "02_nav_history.csv")
aum_df = pd.read_csv(PROCESSED_DIR / "03_aum_by_fund_house.csv")
monthly_sip = pd.read_csv(PROCESSED_DIR / "04_monthly_sip_inflows.csv")
category_inflows = pd.read_csv(PROCESSED_DIR / "05_category_inflows.csv")
industry_folios = pd.read_csv(PROCESSED_DIR / "06_industry_folio_count.csv")
performance = pd.read_csv(PROCESSED_DIR / "07_scheme_performance.csv")
transactions = pd.read_csv(PROCESSED_DIR / "08_investor_transactions.csv")
holdings = pd.read_csv(PROCESSED_DIR / "09_portfolio_holdings.csv")
benchmarks = pd.read_csv(PROCESSED_DIR / "10_benchmark_indices.csv")

print("All 10 processed datasets loaded cleanly.")


In [ ]:
# 1. Daily NAV Trend Analysis across 40 Schemes (2022-2026)
nav_history['date'] = pd.to_datetime(nav_history['date'])

fig = px.line(
    nav_history, 
    x='date', 
    y='nav', 
    color='amfi_code',
    title="Interactive NAV Trend Analysis (2022–2026)",
    labels={'nav': 'Net Asset Value (INR)', 'date': 'Date', 'amfi_code': 'Scheme Code'}
)
fig.add_vrect(x0="2023-03-01", x1="2023-12-31", fillcolor="Green", opacity=0.15, annotation_text="2023 Bull Run")
fig.add_vrect(x0="2024-05-01", x1="2024-10-31", fillcolor="Red", opacity=0.15, annotation_text="2024 Correction")
fig.update_layout(template="plotly_white", height=600)
fig.show()


In [ ]:
# 2. AUM Growth by Fund House
aum_df['year'] = pd.to_datetime(aum_df['date']).dt.year
aum_yearly = aum_df.groupby(['year', 'fund_house'])['aum_crore'].max().reset_index()

plt.figure(figsize=(12, 6))
sns.barplot(data=aum_yearly, x='fund_house', y='aum_crore', hue='year', palette='Blues_r')
plt.title("AUM Growth by Fund House (2022–2025) - Highlighting SBI Dominance", fontsize=14, fontweight='bold')
plt.xticks(rotation=35, ha='right')
plt.ylabel("AUM (INR Crores)")
plt.tight_layout()
plt.show()


In [ ]:
# 3. Monthly SIP Inflow Time-Series
monthly_sip['month'] = pd.to_datetime(monthly_sip['month'])

fig = px.line(
    monthly_sip, 
    x='month', 
    y='sip_inflow_crore',
    markers=True,
    title="Monthly SIP Inflows Trend (Jan 2022 – Dec 2025)",
    labels={'sip_inflow_crore': 'Monthly SIP Inflow (INR Cr)', 'month': 'Month'}
)
fig.add_annotation(
    x="2025-12-01", y=31002,
    text="Dec 2025 All-Time High: ₹31,002 Cr",
    showarrow=True, arrowhead=2, ax=-100, ay=-40,
    font=dict(size=12, color="red")
)
fig.update_layout(template="plotly_white", height=500)
fig.show()


In [ ]:
# 4. Category Inflow Heatmap
cat_pivot = category_inflows.pivot(index='category', columns='month', values='net_inflow_crore')

plt.figure(figsize=(14, 6))
sns.heatmap(cat_pivot, cmap='YlGnBu', annot=False, cbar_kws={'label': 'Net Inflow (INR Crores)'})
plt.title("Monthly Category Net Inflow Heatmap", fontsize=14, fontweight='bold')
plt.xlabel("Month")
plt.ylabel("Category")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# 5. Investor Demographics Analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age Distribution
age_counts = transactions.groupby('age_group')['investor_id'].nunique()
axes[0].pie(age_counts, labels=age_counts.index, autopct='%1.1f%%', colors=sns.color_palette('pastel'))
axes[0].set_title("Age Group Distribution", fontweight='bold')

# SIP Boxplot
sip_txns = transactions[transactions['transaction_type'] == 'SIP']
sns.boxplot(data=sip_txns, x='age_group', y='amount_inr', ax=axes[1], palette='Set2')
axes[1].set_yscale('log')
axes[1].set_title("SIP Amount Distribution by Age (Log Scale)", fontweight='bold')

# Gender Split
gender_counts = transactions.groupby('gender')['investor_id'].nunique()
axes[2].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', colors=['#36A2EB', '#FF6384', '#FFCE56'])
axes[2].set_title("Gender Split", fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# 6. Geographic Distribution & City Tier Split
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# State Investment
state_sum = transactions.groupby('state')['amount_inr'].sum().sort_values(ascending=True) / 1e7
axes[0].barh(state_sum.index, state_sum.values, color='#1f77b4')
axes[0].set_title("Total Investment Value by Indian State (INR Cr)", fontweight='bold')

# City Tier
tier_sum = transactions.groupby('city_tier')['amount_inr'].sum()
axes[1].pie(tier_sum, labels=tier_sum.index, autopct='%1.1f%%', colors=['#4BC0C0', '#FF9F40', '#9966FF'])
axes[1].set_title("City Tier Capital Split (T30 vs B30)", fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# 7. Industry Folio Growth & Milestones
industry_folios['month'] = pd.to_datetime(industry_folios['month'])

plt.figure(figsize=(12, 5))
plt.plot(industry_folios['month'], industry_folios['total_folios_crore'], marker='s', color='#9467bd', label='Total Folios (Cr)', linewidth=2.5)
plt.plot(industry_folios['month'], industry_folios['equity_folios_crore'], linestyle='--', color='#2ca02c', label='Equity Folios (Cr)')
plt.title("Industry Folio Count Growth (Jan 2022: 13.26 Cr → Dec 2025: 26.12 Cr)", fontsize=14, fontweight='bold')
plt.xlabel("Month")
plt.ylabel("Folios (Crores)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 8. NAV Daily Return Correlation Matrix
nav_pivot = nav_history.pivot(index='date', columns='amfi_code', values='nav')
daily_ret = nav_pivot.pct_change().dropna()
top_10 = fund_master['amfi_code'].head(10).tolist()

plt.figure(figsize=(10, 8))
sns.heatmap(daily_ret[top_10].corr(), annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Pairwise Daily NAV Return Correlation Matrix (Top 10 Schemes)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 9. Sector Allocation Donut Chart
sector_weights = holdings.groupby('sector')['weight_pct'].sum().sort_values(ascending=False).head(8)

plt.figure(figsize=(8, 8))
plt.pie(sector_weights, labels=sector_weights.index, autopct='%1.1f%%', startangle=140, 
        colors=sns.color_palette('tab10'), wedgeprops=dict(width=0.45, edgecolor='w'))
plt.title("Aggregated Sector Allocation Across Equity Portfolios", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
